# Kaggle-Ready F5-TTS Mamba3 Training (Multi-GPU T4x2)

Notebook ini disiapkan untuk:
- training F5-TTS dengan backbone `Mamba3Backbone`
- dataset format `metadata.csv + wavs/`
- integrasi `WANDB_API_KEY` dari Kaggle Secret
- multi-GPU training (2x T4) via `accelerate`

Catatan:
- Notebook **tidak** memaksa reinstall PyTorch, supaya tetap kompatibel dengan runtime CUDA bawaan Kaggle (termasuk jika runtime bergeser ke CUDA 13.x).
- Jika internet dibatasi, training tetap jalan karena sample logging otomatis dimatikan saat vocoder download gagal.


## Step 1 - Runtime paths and project settings

Notebook ini sudah disetel untuk source dataset:
`/kaggle/input/datasets/benedictusryugunawan/f5-mamba-dit-tts`

Lalu otomatis copy project ke `/kaggle/working/F5-TTS` (writeable) sebelum training.


In [ ]:
from __future__ import annotations

import csv
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')

# Source dataset path dari request kamu
SOURCE_DATASET_ROOT = Path('/kaggle/input/datasets/benedictusryugunawan/f5-mamba-dit-tts')
SOURCE_DATASET_FALLBACK = Path('/kaggle/input/f5-mamba-dit-tts')


def find_repo_root(base: Path) -> Path | None:
    if not base.exists():
        return None
    if (base / 'pyproject.toml').exists():
        return base
    for child in base.iterdir():
        if child.is_dir() and (child / 'pyproject.toml').exists():
            return child
    return None


source_dataset = SOURCE_DATASET_ROOT if SOURCE_DATASET_ROOT.exists() else SOURCE_DATASET_FALLBACK
SOURCE_REPO_ROOT = find_repo_root(source_dataset) if source_dataset.exists() else None

# Kita kerja di /kaggle/working karena /kaggle/input read-only
REPO_ROOT = KAGGLE_WORKING / 'F5-TTS'

DATA_ROOT = REPO_ROOT / 'data'
RAW_METADATA = DATA_ROOT / 'metadata.csv'
RAW_WAV_DIR = DATA_ROOT / 'wavs'

DATASET_NAME = 'KCVTTS'
TOKENIZER = 'pinyin'
PREPARED_DATA_DIR = REPO_ROOT / 'data' / f'{DATASET_NAME}_{TOKENIZER}'
ABS_METADATA = REPO_ROOT / 'data' / 'metadata_abs.csv'

print('SOURCE_DATASET_ROOT :', SOURCE_DATASET_ROOT)
print('SOURCE_REPO_ROOT    :', SOURCE_REPO_ROOT)
print('REPO_ROOT (working) :', REPO_ROOT)
print('DATA_ROOT           :', DATA_ROOT)
print('RAW_METADATA        :', RAW_METADATA, '| exists =', RAW_METADATA.exists())
print('RAW_WAV_DIR         :', RAW_WAV_DIR, '| exists =', RAW_WAV_DIR.exists())
print('OUTPUT DATASET      :', PREPARED_DATA_DIR)


In [ ]:
if SOURCE_REPO_ROOT is None:
    raise FileNotFoundError(
        'Source repo tidak ditemukan. Cek lagi path dataset Kaggle:\n'
        f'- {SOURCE_DATASET_ROOT}\n'
        f'- {SOURCE_DATASET_FALLBACK}'
    )

if not REPO_ROOT.exists():
    shutil.copytree(SOURCE_REPO_ROOT, REPO_ROOT)
    print('Repo copied to writable path:', REPO_ROOT)
else:
    print('Repo already available at:', REPO_ROOT)

if not (REPO_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(
        f'pyproject.toml tidak ditemukan di {REPO_ROOT}. Pastikan ini root project F5-TTS.'
    )

if not RAW_METADATA.exists() or not RAW_WAV_DIR.exists():
    alt_data_root = SOURCE_REPO_ROOT / 'data'
    alt_meta = alt_data_root / 'metadata.csv'
    alt_wav = alt_data_root / 'wavs'

    if alt_meta.exists() and alt_wav.exists():
        DATA_ROOT = alt_data_root
        RAW_METADATA = alt_meta
        RAW_WAV_DIR = alt_wav
        print('Using fallback read-only DATA_ROOT:', DATA_ROOT)
    else:
        raise FileNotFoundError(
            'Dataset mentah tidak lengkap. Pastikan ada metadata.csv dan folder wavs/ di data/.'
        )

os.chdir(REPO_ROOT)
print('Working directory:', Path.cwd())
print('RAW_METADATA:', RAW_METADATA)
print('RAW_WAV_DIR :', RAW_WAV_DIR)


## Step 2 - Install dependencies (safe for Kaggle CUDA runtime)

Kita install dependency training penting, tanpa memaksa reinstall `torch`.


In [ ]:
def run(cmd: list[str], extra_env: dict[str, str] | None = None) -> None:
    print('+', ' '.join(shlex.quote(c) for c in cmd))
    env = os.environ.copy()

    # Ensure subprocess can import f5_tts from local repo/src.
    src_path = str(REPO_ROOT / 'src')
    old_pythonpath = env.get('PYTHONPATH', '')
    env['PYTHONPATH'] = src_path if not old_pythonpath else f'{src_path}:{old_pythonpath}'

    if extra_env:
        env.update(extra_env)

    subprocess.run(cmd, check=True, env=env)

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])

deps = [
    'accelerate>=1.13.0',
    'datasets',
    'hydra-core>=1.3.0',
    'safetensors',
    'cached_path',
    'pydub',
    'soundfile',
    'librosa',
    'pypinyin',
    'rjieba',
    'ema_pytorch>=0.5.2',
    'torchdiffeq',
    'transformers',
    'transformers_stream_generator',
    'vocos',
    'x_transformers>=1.31.14',
    'wandb',
    'click',
    'tqdm',
]
run([sys.executable, '-m', 'pip', 'install', '-q', *deps])

# Make local package importable in current kernel too.
src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('Dependency setup selesai.')


## Step 3 - GPU/CUDA sanity check

Notebook ini menargetkan multi-GPU T4x2.


In [ ]:
import torch
import accelerate

print('torch version      :', torch.__version__)
print('torch cuda version :', torch.version.cuda)
print('accelerate version :', accelerate.__version__)
print('cuda available     :', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak terdeteksi. Aktifkan GPU Accelerator di Kaggle settings.')

gpu_count = torch.cuda.device_count()
print('gpu count          :', gpu_count)
for i in range(gpu_count):
    print(f'  gpu[{i}]          :', torch.cuda.get_device_name(i))

if gpu_count < 2:
    print('WARNING: GPU terdeteksi < 2. Multi-GPU T4x2 tidak aktif di sesi ini.')
else:
    print('OK: Multi-GPU siap dipakai.')


## Step 4 - WANDB_API_KEY from Kaggle Secret

Buat secret bernama `WANDB_API_KEY` di Kaggle (Add-ons -> Secrets).


In [ ]:
import wandb

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('WANDB_SILENT', 'true')
os.environ.setdefault('WANDB__SERVICE_WAIT', '300')
os.environ.setdefault('WANDB_DIR', str(REPO_ROOT / 'wandb'))

WANDB_ENABLED = False

try:
    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()
    wandb_api_key = secret_client.get_secret('WANDB_API_KEY')
    if wandb_api_key:
        os.environ['WANDB_API_KEY'] = wandb_api_key
        wandb.login(key=wandb_api_key, relogin=True)
        WANDB_ENABLED = True
except Exception as e:
    print('WANDB secret tidak tersedia / gagal dibaca:', e)

if not WANDB_ENABLED:
    os.environ['WANDB_MODE'] = 'offline'

print('WANDB_ENABLED:', WANDB_ENABLED)
print('WANDB_MODE   :', os.getenv('WANDB_MODE', 'online'))


## Step 5 - Convert metadata to absolute path format

`prepare_csv_wavs.py` mengharuskan `audio_file` path absolut.


In [ ]:
rows = []
with RAW_METADATA.open('r', encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f, delimiter='|')
    expected_fields = {'audio_file', 'text'}
    if set(reader.fieldnames or []) != expected_fields:
        raise ValueError(
            f'Header CSV harus tepat audio_file|text, dapat: {reader.fieldnames}'
        )

    for row in reader:
        rel_audio = (row['audio_file'] or '').strip()
        if not rel_audio:
            continue

        rel_path = Path(rel_audio)
        abs_audio = rel_path if rel_path.is_absolute() else (DATA_ROOT / rel_path)
        abs_audio = abs_audio.resolve()

        rows.append({
            'audio_file': abs_audio.as_posix(),
            'text': (row['text'] or '').strip(),
        })

ABS_METADATA.parent.mkdir(parents=True, exist_ok=True)
with ABS_METADATA.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['audio_file', 'text'], delimiter='|')
    writer.writeheader()
    writer.writerows(rows)

print('ABS_METADATA:', ABS_METADATA)
print('Rows written :', len(rows))


## Step 6 - Build F5-TTS training dataset (`raw.arrow`, `duration.json`, `vocab.txt`)


In [ ]:
workers = str(min(8, max(1, (os.cpu_count() or 2) // 2)))
prepare_cmd = [
    sys.executable,
    '-m', 'f5_tts.train.datasets.prepare_csv_wavs',
    str(ABS_METADATA),
    str(PREPARED_DATA_DIR),
    '--pretrain',
    '--workers', workers,
]
run(prepare_cmd)

print('\nPrepared files:')
for p in sorted(PREPARED_DATA_DIR.glob('*')):
    print('-', p.name, '|', p.stat().st_size, 'bytes')


## Step 7 - Smoke test dataset loading


In [ ]:
from f5_tts.model.dataset import load_dataset

dataset = load_dataset(DATASET_NAME, TOKENIZER)
print('Loaded dataset type:', type(dataset).__name__)
print('Dataset length     :', len(dataset))


## Step 8 - Write Kaggle-specific Mamba3 config

Config ini dibuat otomatis agar konsisten dengan dataset kamu.


In [ ]:
config_name = 'kaggle_kcvtts_mamba3.yaml'
config_path = REPO_ROOT / 'src' / 'f5_tts' / 'configs' / config_name
logger_yaml = 'wandb' if WANDB_ENABLED else 'null'

config_text = f"""
hydra:
  run:
    dir: ckpts/${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}/${{now:%Y-%m-%d}}/${{now:%H-%M-%S}}

datasets:
  name: {DATASET_NAME}
  batch_size_per_gpu: 1600
  batch_size_type: frame
  max_samples: 64
  num_workers: 2

optim:
  epochs: 50
  learning_rate: 2e-4
  num_warmup_updates: 4000
  grad_accumulation_steps: 2
  max_grad_norm: 1.0
  bnb_optimizer: False

model:
  name: Mamba3TTS_Base
  tokenizer: {TOKENIZER}
  tokenizer_path: null
  backbone: Mamba3Backbone
  arch:
    dim: 1024
    depth: 22
    ff_mult: 2
    text_dim: 512
    text_mask_padding: True
    conv_layers: 4
    checkpoint_activations: False
    dropout: 0.0
    d_state: 128
    headdim: 64
    ngroups: 1
    rope_fraction: 0.5
    is_mimo: False
    mimo_rank: 1
    chunk_size: 64
  mel_spec:
    target_sample_rate: 24000
    n_mel_channels: 100
    hop_length: 256
    win_length: 1024
    n_fft: 1024
    mel_spec_type: vocos
  vocoder:
    is_local: False
    local_path: null

ckpts:
  logger: {logger_yaml}
  wandb_project: Mamba3-TTS
  wandb_run_name: ${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}
  wandb_resume_id: null
  log_samples: false
  save_per_updates: 10000
  keep_last_n_checkpoints: 3
  last_per_updates: 2000
  save_dir: ckpts/${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}
""".strip() + "\n"

config_path.write_text(config_text, encoding='utf-8')
print('Training config written to:', config_path)
print(config_path.read_text(encoding='utf-8').splitlines()[:12])


## Step 9 - Write Accelerate config for T4x2 (DDP)


In [ ]:
num_gpus = torch.cuda.device_count()
num_processes = 2 if num_gpus >= 2 else 1
distributed_type = 'MULTI_GPU' if num_processes > 1 else 'NO'

accelerate_cfg_path = REPO_ROOT / 'accelerate_kaggle.yaml'
accelerate_cfg_text = f"""
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: {distributed_type}
downcast_bf16: 'no'
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: {num_processes}
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
""".strip() + "\n"

accelerate_cfg_path.write_text(accelerate_cfg_text, encoding='utf-8')
print('Accelerate config:', accelerate_cfg_path)
print(accelerate_cfg_path.read_text(encoding='utf-8'))


## Step 10 - Launch multi-GPU training

Cell ini akan memulai training. Jalankan saat semua step sebelumnya sudah sukses.


In [ ]:
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if torch.cuda.device_count() >= 2:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

train_cmd = [
    sys.executable,
    '-m', 'accelerate.commands.launch',
    '--config_file', str(accelerate_cfg_path),
    'src/f5_tts/train/train.py',
    '--config-name', config_name,
]
run(train_cmd)


## Step 11 - Resume training (optional)

Jika sesi Kaggle putus, jalankan lagi cell launch training.
Trainer otomatis membaca checkpoint terbaru di folder `ckpts/...`.

Tips stabilitas:
- Jika OOM: turunkan `datasets.batch_size_per_gpu` (contoh: 1200 atau 800).
- Jika data loader error: turunkan `datasets.num_workers` ke `0`.
- Jika W&B secret tidak ada: training tetap jalan mode offline.
